# 02 - Model A

Latih model transkripsi CNN + BiGRU + CTC menggunakan data lokal. Training penuh ditujukan untuk Colab T4; GPU lokal 4 GB mungkin memerlukan penyesuaian batch.

Jalankan sampai **Training dan validasi** untuk percobaan awal. Cek sampel per detik pada epoch kedua, setelah cache fitur terisi. Jalankan **Test Model A** setelah konfigurasi dipilih. Jika sesi putus, gunakan RUN_ID yang sama untuk melanjutkan dari checkpoint terakhir.

Checkpoint disimpan di `checkpoints/model_a` dan hasil di `results`. Salin checkpoint dari Drive jika ingin melanjutkan run Colab.

## Konfigurasi Model A

| Pengaturan | Nilai |
| --- | --- |
| Dataset | Google FLEURS Indonesia (`id_id`) dan Inggris (`en_us`) |
| Data training | 5.181 sampel |
| Data validation | 744 sampel |
| Sample rate | 16.000 Hz |
| Fitur audio | Log-Mel spectrogram, 80 Mel bins |
| FFT / hop length | 400 / 160 sampel |
| Kosakata | Karakter dari transkrip train, termasuk spasi dan blank CTC |
| Arsitektur | 2 CNN + 2 BiGRU + linear CTC |
| CNN 1 | Conv1d, kanal 80 → 64, kernel 5, stride 2 |
| CNN 2 | Conv1d, kanal 64 → 128, kernel 3, stride 1 |
| Aktivasi CNN | ReLU |
| BiGRU | 2 layer, 256 unit per arah, dropout 0,1 |
| Output | Linear dan log-softmax |
| Jumlah parameter | 1.854.199 |
| Loss | CTC loss |
| Optimizer | AdamW |
| Learning rate | 0,001 |
| Batch size | 32 |
| Epoch | Maksimum 30 (lanjutan dari epoch 20) |
| Gradient clipping | Maksimum 5,0 |
| Data loader | 0 worker (stabilitas Colab) |
| Cache fitur | Folder lokal `data/preprocessed/model_a` |
| Decoding | CTC greedy decoding |
| Pemilihan checkpoint | Rata-rata WER dev terendah dari kedua bahasa |
| GPU | NVIDIA T4 jika tersedia |
| RUN_ID | `model_a` |
| Resume | Aktif, melanjutkan dari `latest.pt` |


## Pengaturan

Pilih mode, data, dan perangkat untuk menjalankan model.

In [ ]:
from pathlib import Path
from collections import defaultdict
import csv
import hashlib
import json
import math
import os
import random
import time
import unicodedata

import numpy as np
from scipy.io import wavfile
from scipy.signal import resample_poly
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from torch.utils.data import DataLoader, Dataset

RUN_ID = "model_a"
RESUME = True
SEED = 42
BATCH_SIZE = 32
EPOCHS = 35
LEARNING_RATE = 1e-3
LANGUAGES = ("id_id", "en_us")
SAMPLE_RATE, N_FFT, HOP, N_MELS = 16000, 400, 160, 80
MODEL_CONFIG = {"hidden": 256, "layers": 2, "channels": [64, 128]}
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_ROOT = root / "data"
DRIVE_DATA_ROOT = DATA_ROOT
TRAINING_ROOT = root
RUN_DIR = root / "checkpoints" / RUN_ID
RESULTS_DIR = root / "results"
pilot_passed = False
selected_checkpoint = None

print(f"Perangkat: {torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'}")
print(f"Folder data: {DATA_ROOT}")
print(f"Checkpoint: {RUN_DIR}")

## Persiapan data

Baca transkrip dan ubah WAV menjadi fitur log-Mel.

In [ ]:
def normalize_text(text):
    chars = unicodedata.normalize("NFC", text).casefold()
    chars = chars.replace("’", "'")
    return " ".join("".join(c if c.isalnum() or c == "'" else " " for c in chars).split())


def read_split(root, split):
    rows = []
    for language in LANGUAGES:
        locale = root / language
        path = locale / (split + ".tsv")
        with path.open(encoding="utf-8-sig", newline="") as stream:
            for number, fields in enumerate(csv.reader(stream, delimiter=chr(9), quoting=csv.QUOTE_NONE), 1):
                if not any(fields):
                    continue
                if len(fields) != 7 or not fields[0].isdigit():
                    raise ValueError(f"{path}:{number}: format TSV tidak valid.")
                sentence_id, filename, _, normalized, *_ = fields
                text = normalize_text(normalized)
                if Path(filename).name != filename or "/" in filename or chr(92) in filename or not filename.lower().endswith(".wav") or not text:
                    raise ValueError(f"{path}:{number}: nama WAV atau transkrip tidak valid.")
                audio = locale / "audio" / split / filename
                rows.append((language, audio, text, sentence_id))
        if not rows or rows[-1][0] != language:
            raise ValueError(f"Tidak ada sampel {language}/{split}.")
    keys = [(language, audio.name) for language, audio, *_ in rows]
    if len(set(keys)) != len(keys):
        raise ValueError(f"Ada nama WAV ganda dalam split {split}. Tinjau audit data.")
    return rows


def pilot_subset(rows, per_language):
    return [row for language in LANGUAGES for row in
            [item for item in rows if item[0] == language][:per_language]]


def make_mel_bank():
    def hz_to_mel(hz):
        return 2595 * math.log10(1 + hz / 700)

    mel_points = torch.linspace(hz_to_mel(0), hz_to_mel(SAMPLE_RATE / 2), N_MELS + 2)
    left, center, right = (700 * (10 ** (mel_points / 2595) - 1)).unfold(0, 3, 1).T
    fft_hz = torch.linspace(0, SAMPLE_RATE / 2, N_FFT // 2 + 1)
    rising = (fft_hz - left[:, None]) / (center - left)[:, None]
    falling = (right[:, None] - fft_hz) / (right - center)[:, None]
    return torch.minimum(rising, falling).clamp_min(0)


MEL_BANK = make_mel_bank()
WINDOW = torch.hann_window(N_FFT)


def audio_features(path):
    if not path.is_file():
        raise FileNotFoundError(f"{path} belum ada. Ekstrak arsip WAV sebelum menjalankan notebook.")
    rate, pcm = wavfile.read(path)
    samples = pcm.astype(np.float32)
    if np.issubdtype(pcm.dtype, np.integer):
        info = np.iinfo(pcm.dtype)
        if info.min == 0:
            midpoint = (info.max + 1) / 2
            samples = (samples - midpoint) / midpoint
        else:
            samples /= -info.min
    if samples.ndim == 2:
        samples = samples.mean(axis=1)
    if rate <= 0 or samples.size == 0 or not np.isfinite(samples).all():
        raise ValueError(f"Audio kosong atau tidak valid: {path}")
    if rate != SAMPLE_RATE:
        divisor = math.gcd(rate, SAMPLE_RATE)
        samples = resample_poly(samples, SAMPLE_RATE // divisor, rate // divisor)
    signal = torch.from_numpy(samples)
    power = torch.stft(signal, N_FFT, HOP, window=WINDOW, pad_mode="constant",
                       return_complex=True).abs().square()
    mel = (MEL_BANK @ power).clamp_min(1e-10).log()
    return mel.T.contiguous()


FEATURE_CACHE = root / "data" / "preprocessed" / "model_a"


def cached_audio_features(path):
    relative = path.relative_to(DATA_ROOT)
    cache_dir = FEATURE_CACHE / relative.parent
    signature = f"{SAMPLE_RATE}-{N_FFT}-{HOP}-{N_MELS}-{path.stat().st_mtime_ns}"
    cache_path = cache_dir / f"{path.stem}-{signature}.npy"

    if cache_path.is_file():
        return torch.from_numpy(np.load(cache_path, allow_pickle=False))

    features = audio_features(path)
    cache_dir.mkdir(parents=True, exist_ok=True)
    temporary = cache_path.with_suffix(f".{os.getpid()}.tmp")
    with temporary.open("wb") as file:
        np.save(file, features.numpy(), allow_pickle=False)
    temporary.replace(cache_path)
    return features


class SpeechDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        language, path, text, _ = self.rows[index]
        features = (
            cached_audio_features(path)
            if path.is_relative_to(DATA_ROOT)
            else audio_features(path)
        )
        return features, language, text, path.name


def collate(items):
    features, languages, texts, names = zip(*items)
    lengths = torch.tensor([len(item) for item in features], dtype=torch.long)
    return pad_sequence(features, batch_first=True), lengths, languages, texts, names


def loader(rows, shuffle=False, seed=SEED):
    return DataLoader(SpeechDataset(rows), batch_size=BATCH_SIZE, shuffle=shuffle,
                      generator=torch.Generator().manual_seed(seed),
                      collate_fn=collate, num_workers=0, pin_memory=DEVICE.type == "cuda")

## Model dan metrik

Buat CNN + BiGRU + CTC dan hitung WER serta CER.

In [ ]:
class ModelA(nn.Module):
    def __init__(self, vocab_size, hidden, layers, channels):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv1d(N_MELS, channels[0], kernel_size=5, stride=2, padding=2),
            nn.Conv1d(channels[0], channels[1], kernel_size=3, stride=1, padding=1),
        ])
        self.gru = nn.GRU(channels[1], hidden, num_layers=layers,
                          batch_first=True, bidirectional=True,
                          dropout=0.1 if layers > 1 else 0)
        self.projection = nn.Linear(hidden * 2, vocab_size)

    def forward(self, features, lengths):
        features = features.transpose(1, 2)
        for conv in self.convs:
            features = torch.relu(conv(features))
            lengths = (lengths + 2 * conv.padding[0] - conv.kernel_size[0]) // conv.stride[0] + 1
            valid = torch.arange(features.size(-1), device=features.device) < lengths.to(features.device)[:, None]
            features = features.masked_fill(~valid[:, None, :], 0)
        packed = pack_padded_sequence(features.transpose(1, 2), lengths,
                                      batch_first=True, enforce_sorted=False)
        features, _ = pad_packed_sequence(self.gru(packed)[0], batch_first=True)
        return self.projection(features).log_softmax(-1), lengths


def vocabulary(rows):
    chars = sorted({char for _, _, text, _ in rows for char in text})
    return ["<blank>", *chars]


def training_loss(model, batch, char_to_id):
    features, lengths, _, texts, names = batch
    predictions, output_lengths = model(features.to(DEVICE, non_blocking=True), lengths)
    targets = [[char_to_id[char] for char in text] for text in texts]
    for target, length, filename in zip(targets, output_lengths.tolist(), names):
        required = len(target) + sum(a == b for a, b in zip(target, target[1:]))
        if length < required:
            raise ValueError(f"CTC tidak bisa menyelaraskan {filename}: {length} frame, perlu {required}.")
    joined = torch.tensor([token for sequence in targets for token in sequence], device=DEVICE)
    target_lengths = torch.tensor([len(sequence) for sequence in targets])
    return nn.functional.ctc_loss(predictions.transpose(0, 1), joined, output_lengths, target_lengths)


def decode(token_ids, vocab):
    chars = []
    previous = None
    for token in token_ids:
        if token != 0 and token != previous:
            chars.append(vocab[token])
        previous = token
    return "".join(chars)


def edit_distance(reference, hypothesis):
    previous = list(range(len(hypothesis) + 1))
    for i, expected in enumerate(reference, 1):
        current = [i]
        for j, actual in enumerate(hypothesis, 1):
            current.append(min(current[-1] + 1, previous[j] + 1,
                               previous[j - 1] + (expected != actual)))
        previous = current
    return previous[-1]


@torch.inference_mode()
def evaluate(model, vocab, rows, progress_label=None):
    model.eval()
    counts = defaultdict(lambda: [0, 0, 0, 0])
    batches = loader(rows)
    if progress_label:
        batches = progress_batches(batches, progress_label)
    for features, lengths, languages, texts, _ in batches:
        predictions, output_lengths = model(features.to(DEVICE, non_blocking=True), lengths)
        ids = predictions.argmax(-1).cpu().tolist()
        for tokens, length, language, reference in zip(ids, output_lengths.tolist(), languages, texts):
            hypothesis = normalize_text(decode(tokens[:length], vocab))
            words = reference.split()
            counts[language][0] += edit_distance(words, hypothesis.split())
            counts[language][1] += len(words)
            counts[language][2] += edit_distance(reference, hypothesis)
            counts[language][3] += len(reference)
    per_language = {lang: {"wer": w / n, "cer": c / m, "reference_words": n}
                    for lang, (w, n, c, m) in counts.items()}
    return {"per_language": per_language,
            "macro_wer": sum(per_language[lang]["wer"] for lang in LANGUAGES) / len(LANGUAGES)}

## Tampilan hasil

Siapkan tabel dan grafik untuk membaca hasil eksperimen.

In [ ]:
from io import BytesIO

from IPython.display import Image, Markdown, display
from matplotlib.figure import Figure
from matplotlib.ticker import MaxNLocator, PercentFormatter

LANGUAGE_NAMES = {"id_id": "Indonesia", "en_us": "Inggris"}
LANGUAGE_COLORS = {"id_id": "#007C83", "en_us": "#6D54AD"}


def show_metrics(title, metrics):
    lines = [f"### {title}", "", "| Bahasa | WER | CER | Kata acuan |",
             "| --- | ---: | ---: | ---: |"]
    for language in LANGUAGES:
        values = metrics["per_language"][language]
        lines.append(f"| {LANGUAGE_NAMES[language]} | {values['wer']:.1%} | "
                     f"{values['cer']:.1%} | {values['reference_words']:,} |")
    lines += ["", f"**Rata-rata WER dua bahasa:** {metrics['macro_wer']:.1%}."]
    if "train_loss" in metrics:
        lines.append(f"**Waktu total epoch:** {metrics['elapsed_seconds']:.1f} detik.")
    if metrics.get("peak_vram_mib") is not None:
        lines.append(f"**Puncak VRAM PyTorch:** {metrics['peak_vram_mib']:,.0f} MiB.")
    display(Markdown("\n".join(lines)))


def show_figure(figure):
    with BytesIO() as buffer:
        figure.savefig(buffer, format="png", dpi=130)
        display(Image(data=buffer.getvalue(), format="png"))


def show_error_bars(metrics, split_name):
    figure = Figure(figsize=(7, 3), layout="constrained")
    ax = figure.subplots()
    positions = list(range(len(LANGUAGES)))
    for metric, offset, color in (("wer", -0.17, "#007C83"), ("cer", 0.17, "#6D54AD")):
        rates = [metrics["per_language"][lang][metric] for lang in LANGUAGES]
        bars = ax.bar([p + offset for p in positions], rates, width=0.3,
                      label=metric.upper(), color=color)
        ax.bar_label(bars, labels=[f"{rate:.1%}" for rate in rates], padding=3)
    ax.set_xticks(positions, [LANGUAGE_NAMES[lang] for lang in LANGUAGES])
    ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.set_ylim(0, max(0.1, ax.get_ylim()[1] * 1.18))
    ax.set_title(f"Error transkripsi | {split_name}")
    ax.legend()
    show_figure(figure)


def show_training_history(history):
    if not history:
        return
    figure = Figure(figsize=(11, 3), layout="constrained")
    loss_ax, wer_ax = figure.subplots(1, 2)
    epochs = [item["epoch"] for item in history]
    loss_ax.plot(epochs, [item["train_loss"] for item in history], "o-", color="#007C83")
    loss_ax.set_title("Loss training (CTC)")
    for language in LANGUAGES:
        wer_ax.plot(epochs, [item["per_language"][language]["wer"] for item in history],
                    "o-", label=LANGUAGE_NAMES[language], color=LANGUAGE_COLORS[language])
    best_epoch = min(history, key=lambda item: item["macro_wer"])["epoch"]
    wer_ax.axvline(best_epoch, linestyle=":", color="#555555", label=f"Terbaik: epoch {best_epoch}")
    wer_ax.set_title("WER pada data dev")
    wer_ax.yaxis.set_major_formatter(PercentFormatter(1))
    wer_ax.legend()
    for ax in (loss_ax, wer_ax):
        ax.set_xlabel("Epoch")
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.grid(axis="y", alpha=0.25)
    show_figure(figure)

## Alur eksperimen

Siapkan training, validasi, checkpoint, dan test.

In [ ]:
from IPython.display import HTML
import html


def progress_batches(batches, label):
    total = len(batches)
    started = time.perf_counter()
    last_update = started
    completed = 0
    safe_label = html.escape(label)

    def render(status):
        percent = 100 * completed / total if total else 100
        elapsed = time.perf_counter() - started
        return HTML(
            f"<div style='margin:8px 0'><strong>{safe_label}</strong> "
            f"— {status} · {completed}/{total} batch · {percent:.0f}% · "
            f"{elapsed:.1f} detik"
            f"<div style='height:10px;background:#e5e7eb;border-radius:5px;overflow:hidden'>"
            f"<div style='height:100%;width:{percent:.1f}%;background:#008b83'></div>"
            f"</div></div>"
        )

    progress = display(render("berjalan"), display_id=True)
    try:
        for batch in batches:
            yield batch
            completed += 1
            now = time.perf_counter()
            if now - last_update >= 0.5:
                progress.update(render("berjalan"))
                last_update = now
    finally:
        status = "selesai" if completed == total else "terhenti"
        progress.update(render(status))


def save_checkpoint(path, state):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    torch.save(state, temporary)
    temporary.replace(path)


def save_metrics(path, metrics):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".tmp")
    temporary.write_text(json.dumps(metrics, indent=2) + chr(10), encoding="utf-8")
    temporary.replace(path)


def split_fingerprint(rows):
    serialized = json.dumps([(lang, path.name, text, sid) for lang, path, text, sid in rows],
                            ensure_ascii=False)
    return hashlib.sha256(serialized.encode("utf-8")).hexdigest()


def run_training(pilot):
    train_rows = read_split(DATA_ROOT, "train")
    dev_rows = read_split(DATA_ROOT, "dev")
    vocab = vocabulary(train_rows)
    char_to_id = {char: index for index, char in enumerate(vocab)}
    settings = {"model": MODEL_CONFIG, "vocab": vocab, "seed": SEED, "batch": BATCH_SIZE,
                "learning_rate": LEARNING_RATE, "train": split_fingerprint(train_rows),
                "dev": split_fingerprint(dev_rows),
                "features": [SAMPLE_RATE, N_FFT, HOP, N_MELS]}
    if pilot:
        train_rows = pilot_subset(train_rows, 2)
        dev_rows = pilot_subset(dev_rows, 4)

    random.seed(SEED)
    torch.manual_seed(SEED)
    model = ModelA(len(vocab), **MODEL_CONFIG).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    print("Parameter:", sum(p.numel() for p in model.parameters()))
    print(f"Data: {len(train_rows):,} train | {len(dev_rows):,} dev")
    if pilot:
        print("Pilot memakai sedikit sampel untuk mengecek alur training.")
    latest = RUN_DIR / "latest.pt"
    best = RUN_DIR / "best.pt"
    best_score, start_epoch = float("inf"), 1
    if not pilot and not latest.exists() and (RESULTS_DIR / "validation" / RUN_ID).exists():
        raise FileNotFoundError(
            f"Checkpoint latest.pt tidak ditemukan untuk {RUN_ID}. "
            "Periksa Drive sebelum memulai ulang."
        )
    if not pilot and latest.exists():
        if not best.is_file():
            raise FileNotFoundError(
                f"latest.pt ada, tetapi best.pt tidak ditemukan di {RUN_DIR}. "
                "Periksa checkpoint sebelum melanjutkan."
            )
        if not RESUME:
            raise ValueError("RUN_ID sudah dipakai. Gunakan RUN_ID baru atau aktifkan RESUME.")
        state = torch.load(latest, map_location=DEVICE, weights_only=True)
        if state["settings"] != settings:
            raise ValueError("Dataset atau konfigurasi berubah. Gunakan RUN_ID baru.")
        model.load_state_dict(state["model"])
        optimizer.load_state_dict(state["optimizer"])
        best_score, start_epoch = state["best_score"], state["epoch"] + 1
        print("Melanjutkan dari epoch", start_epoch)

    history = []
    validation_dir = RESULTS_DIR / "validation" / RUN_ID
    if start_epoch > 1:
        for path in sorted(validation_dir.glob("epoch_*.json")):
            if int(path.stem.rsplit("_", 1)[1]) < start_epoch:
                history.append(json.loads(path.read_text(encoding="utf-8")))
    for epoch in range(start_epoch, (1 if pilot else EPOCHS) + 1):
        total_epochs = 1 if pilot else EPOCHS
        display(Markdown(f"---\n## Epoch {epoch} / {total_epochs}\n### Training"))
        model.train()
        torch.manual_seed(SEED + epoch)
        batches = loader(train_rows, shuffle=True, seed=SEED + epoch)
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        started = time.perf_counter()
        losses = []
        for step, batch in enumerate(progress_batches(batches, "Training"), 1):
            optimizer.zero_grad(set_to_none=True)
            loss = training_loss(model, batch, char_to_id)
            if not torch.isfinite(loss).item():
                raise RuntimeError(f"Loss tidak valid pada epoch {epoch}, batch {step}.")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0, error_if_nonfinite=True)
            optimizer.step()
            losses.append(loss.item())
        train_seconds = time.perf_counter() - started
        display(Markdown(
            f"**Training selesai** · Loss rata-rata: {sum(losses) / len(losses):.3f} · "
            f"Waktu: {train_seconds:.1f} detik · "
            f"Kecepatan: {len(train_rows) / train_seconds:.1f} sampel/detik"
        ))
        display(Markdown("### Validasi dev"))
        metrics = evaluate(model, vocab, dev_rows, progress_label="Validasi dev")
        metrics.update(epoch=epoch, train_loss=sum(losses) / len(losses),
                       train_seconds=round(train_seconds, 2),
                       train_samples_per_second=round(len(train_rows) / train_seconds, 2),
                       elapsed_seconds=round(time.perf_counter() - started, 2),
                       peak_vram_mib=round(torch.cuda.max_memory_allocated() / 2**20, 1)
                       if DEVICE.type == "cuda" else None, device=str(DEVICE))
        history.append(metrics)
        show_metrics("Hasil dev pilot" if pilot else "Hasil dev", metrics)
        if not pilot:
            display(Markdown("### Contoh prediksi dev"))
            model.eval()
            with torch.inference_mode():
                examples = pilot_subset(dev_rows, 1)
                for features, lengths, languages, texts, _ in loader(examples):
                    predictions, output_lengths = model(
                        features.to(DEVICE, non_blocking=True), lengths
                    )
                    tokens = predictions.argmax(-1).cpu().tolist()
                    for language, reference, ids, length in zip(
                        languages, texts, tokens, output_lengths.tolist()
                    ):
                        print(f"{language} acuan: {reference}")
                        print(f"{language} prediksi: {decode(ids[:length], vocab)!r}")
            improved = metrics["macro_wer"] < best_score
            best_score = min(best_score, metrics["macro_wer"])
            state = {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                     "settings": settings, "epoch": epoch, "best_score": best_score}
            save_metrics(validation_dir / f"epoch_{epoch:03}.json", metrics)
            if improved:
                save_checkpoint(best, state)
            save_checkpoint(latest, state)
            if not latest.is_file() or not best.is_file():
                raise FileNotFoundError(
                    f"Checkpoint belum lengkap di {RUN_DIR}. "
                    "Periksa best.pt dan latest.pt sebelum melanjutkan."
                )
            print(f"Checkpoint epoch {epoch} tersimpan di {RUN_DIR}", flush=True)
    if pilot:
        show_error_bars(history[-1], "Dev pilot")
    else:
        show_training_history(history)
        if history:
            best_metrics = min(history, key=lambda item: item["macro_wer"])
            show_metrics(f"Checkpoint terbaik | epoch {best_metrics['epoch']}", best_metrics)
        print("Checkpoint terbaik di dev:", best)
    return model, vocab


def run_final_test(checkpoint):
    if not checkpoint.is_file():
        raise FileNotFoundError(checkpoint)
    state = torch.load(checkpoint, map_location=DEVICE, weights_only=True)
    vocab = state["settings"]["vocab"]
    model = ModelA(len(vocab), **state["settings"]["model"]).to(DEVICE)
    model.load_state_dict(state["model"])
    metrics = evaluate(model, vocab, read_split(DRIVE_DATA_ROOT, "test"))
    metrics.update(checkpoint=str(checkpoint), checkpoint_epoch=state["epoch"])
    destination = RESULTS_DIR / "test" / checkpoint.parent.name / "model_a.json"
    save_metrics(destination, metrics)
    show_metrics(f"Test akhir | checkpoint epoch {state['epoch']}", metrics)
    show_error_bars(metrics, "Test akhir")
    print("Laporan:", destination)
    return model, vocab

## Pilot

Latih sebentar dengan beberapa sampel untuk mengecek apakah audio, model, dan evaluasi berjalan.

In [ ]:
pilot_passed = False
selected_checkpoint = None
run_training(pilot=True)
pilot_passed = True
print("Pilot selesai.")

## Training dan validasi

Latih dengan data train. Checkpoint dengan rata-rata WER dev terendah disimpan sebagai best.pt.

In [ ]:
selected_checkpoint = None
if not pilot_passed:
    raise RuntimeError("Jalankan pilot sampai berhasil sebelum training.")
if EPOCHS < 1:
    raise ValueError("EPOCHS harus minimal 1.")

model, vocab = run_training(pilot=False)
checkpoint = RUN_DIR / "best.pt"
if not checkpoint.is_file():
    raise FileNotFoundError(f"Checkpoint terbaik belum tersedia: {checkpoint}")
selected_checkpoint = checkpoint

## Test Model A

Ukur WER dan CER pada data test dengan checkpoint yang sudah dipilih di dev. Gunakan hasil ini untuk laporan.

In [ ]:
if selected_checkpoint is None:
    raise RuntimeError("Selesaikan training dan validasi sebelum menjalankan test.")

model, vocab = run_final_test(selected_checkpoint)

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

# Muat checkpoint terbaik berdasarkan rata-rata WER dev.
checkpoint = RUN_DIR / "best.pt"

if not checkpoint.is_file():
    raise FileNotFoundError(f"Checkpoint tidak ditemukan: {checkpoint}")

state = torch.load(checkpoint, map_location=DEVICE, weights_only=True)
best_epoch = state["epoch"]

eval_vocab = state["settings"]["vocab"]
eval_model = ModelA(
    len(eval_vocab),
    **state["settings"]["model"],
).to(DEVICE)

eval_model.load_state_dict(state["model"])
eval_model.eval()

# Baca riwayat dev.
history_dir = RESULTS_DIR / "validation" / RUN_ID
history_files = sorted(history_dir.glob("epoch_*.json"))

if not history_files:
    raise FileNotFoundError(f"Riwayat dev tidak ditemukan: {history_dir}")

history = [
    json.loads(path.read_text(encoding="utf-8"))
    for path in history_files
]
history.sort(key=lambda item: item["epoch"])

# Hitung metrik train dan test untuk checkpoint terbaik.
print(f"Checkpoint terbaik: epoch {best_epoch}")
print("Menghitung hasil train...", flush=True)

train_metrics = evaluate(
    eval_model,
    eval_vocab,
    read_split(DATA_ROOT, "train"),
    progress_label="Evaluasi train",
)

print("Menghitung hasil test...", flush=True)

test_metrics = evaluate(
    eval_model,
    eval_vocab,
    read_split(DRIVE_DATA_ROOT, "test"),
    progress_label="Evaluasi test",
)

# Tabel hasil train dan test.
lines = [
    f"## Hasil checkpoint terbaik — epoch {best_epoch}",
    "",
    "| Data | Bahasa | WER | CER |",
    "| --- | --- | ---: | ---: |",
]

for split_name, metrics in (
    ("Train", train_metrics),
    ("Test", test_metrics),
):
    for language in ("id_id", "en_us"):
        values = metrics["per_language"][language]
        lines.append(
            f"| {split_name} | {LANGUAGE_NAMES[language]} | "
            f"{values['wer']:.1%} | {values['cer']:.1%} |"
        )

display(Markdown("\n".join(lines)))

# Grafik dev per epoch.
fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")

epochs = [item["epoch"] for item in history]
colors = {"id_id": "#007C83", "en_us": "#6D54AD"}

for language in ("id_id", "en_us"):
    for metric, style in (("wer", "-"), ("cer", "--")):
        values = [
            item["per_language"][language][metric] * 100
            for item in history
        ]
        ax.plot(
            epochs,
            values,
            linestyle=style,
            color=colors[language],
            label=f"{LANGUAGE_NAMES[language]} {metric.upper()}",
        )

ax.axvline(
    best_epoch,
    color="#555555",
    linestyle=":",
    label=f"Checkpoint terbaik: epoch {best_epoch}",
)
ax.set_title("WER dan CER dev per epoch")
ax.set_xlabel("Epoch")
ax.set_ylabel("Error (%)")
ax.grid(alpha=0.25)
ax.legend()

plt.show()

# Grafik batang train dan test pada checkpoint terbaik.
labels = [
    "Indonesia\nWER",
    "Indonesia\nCER",
    "Inggris\nWER",
    "Inggris\nCER",
]

train_values = [
    train_metrics["per_language"][language][metric] * 100
    for language in ("id_id", "en_us")
    for metric in ("wer", "cer")
]
test_values = [
    test_metrics["per_language"][language][metric] * 100
    for language in ("id_id", "en_us")
    for metric in ("wer", "cer")
]

positions = np.arange(len(labels))
width = 0.34

fig, ax = plt.subplots(figsize=(10, 4), layout="constrained")

train_bars = ax.bar(
    positions - width / 2,
    train_values,
    width,
    label="Train",
    color="#007C83",
)
test_bars = ax.bar(
    positions + width / 2,
    test_values,
    width,
    label="Test",
    color="#6D54AD",
)

ax.bar_label(train_bars, fmt="%.1f%%", padding=3)
ax.bar_label(test_bars, fmt="%.1f%%", padding=3)
ax.set_xticks(positions, labels)
ax.set_title(f"Train dan test — checkpoint epoch {best_epoch}")
ax.set_ylabel("Error (%)")
ax.set_ylim(0, max(train_values + test_values + [1]) * 1.2)
ax.grid(axis="y", alpha=0.25)
ax.set_axisbelow(True)
ax.legend()

plt.show()


In [ ]:
# Cari contoh prediksi terbaik dan terburuk di data dev.

from IPython.display import Markdown, display

dev_rows = read_split(DATA_ROOT, "dev")
examples = {language: [] for language in ("id_id", "en_us")}

print("Mencari contoh prediksi dev...", flush=True)

eval_model.eval()

with torch.inference_mode():
    batches = progress_batches(loader(dev_rows), "Memeriksa data dev")

    for features, lengths, languages, texts, names in batches:
        predictions, output_lengths = eval_model(
            features.to(DEVICE, non_blocking=True),
            lengths,
        )
        token_ids = predictions.argmax(-1).cpu().tolist()

        for language, reference, tokens, length, filename in zip(
            languages,
            texts,
            token_ids,
            output_lengths.tolist(),
            names,
        ):
            prediction = normalize_text(
                decode(tokens[:length], eval_vocab)
            )
            reference = normalize_text(reference)
            reference_words = reference.split()

            if not reference_words:
                continue

            errors = edit_distance(reference_words, prediction.split())
            examples[language].append({
                "wer": errors / len(reference_words),
                "reference": reference,
                "prediction": prediction,
                "filename": filename,
            })

display(Markdown(f"## Contoh prediksi dev — epoch {best_epoch}"))

for language in ("id_id", "en_us"):
    language_examples = examples[language]

    if not language_examples:
        print(f"Tidak ada contoh untuk {LANGUAGE_NAMES[language]}.")
        continue

    best_example = min(language_examples, key=lambda item: item["wer"])
    worst_example = max(language_examples, key=lambda item: item["wer"])

    display(Markdown(f"### {LANGUAGE_NAMES[language]}"))

    for title, item in (
        ("Terbaik", best_example),
        ("Terburuk", worst_example),
    ):
        display(Markdown(f"**{title} — WER {item['wer']:.1%}**"))
        print(f"Audio    : {item['filename']}")
        print(f"Acuan    : {item['reference']}")
        print(f"Prediksi : {item['prediction'] or '(kosong)'}")
        print()

print("Selesai.")


## Ringkasan laporan (metrik, parameter, waktu training, dan kurva)

Menampilkan jumlah parameter, rata-rata dan total waktu training, metrik checkpoint terbaik, serta kurva training loss dan validasi dev dari riwayat yang tersimpan tanpa melatih ulang.

In [ ]:
# Ringkasan laporan Model A (metrik, parameter, waktu training, kurva, & titik Test Epoch 25)
# Sel ini mandiri: membaca checkpoint & riwayat validasi lokal tanpa melatih ulang.
from pathlib import Path
import csv
import json
import math
import unicodedata
import numpy as np
from scipy.io import wavfile
from scipy.signal import resample_poly
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import torch
from torch import nn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from IPython.display import Markdown, display

_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RUN_ID = globals().get("RUN_ID", "model_a")
RUN_DIR = globals().get("RUN_DIR", _root / "checkpoints" / RUN_ID)
RESULTS_DIR = globals().get("RESULTS_DIR", _root / "results")
DRIVE_DATA_ROOT = globals().get("DRIVE_DATA_ROOT", _root / "data")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

checkpoint = RUN_DIR / "best.pt"
history_dir = RESULTS_DIR / "validation" / RUN_ID
history_files = sorted(history_dir.glob("epoch_*.json"))

if not checkpoint.is_file():
    raise FileNotFoundError(f"Checkpoint terbaik tidak ditemukan: {checkpoint}")
if not history_files:
    raise FileNotFoundError(f"Riwayat validasi per epoch tidak ditemukan: {history_dir}")

state = torch.load(checkpoint, map_location="cpu", weights_only=True)
best_epoch = state["epoch"]
total_params = sum(tensor.numel() for tensor in state["model"].values())
trainable_params = total_params

history = [json.loads(p.read_text(encoding="utf-8")) for p in history_files]
history.sort(key=lambda item: item["epoch"])
best_dev = next(
    (item for item in history if item["epoch"] == best_epoch),
    min(history, key=lambda item: item["macro_wer"]),
)

test_json = RESULTS_DIR / "test" / RUN_ID / "model_a.json"
test_saved = json.loads(test_json.read_text(encoding="utf-8")) if test_json.is_file() else None

test_loss_cache = RESULTS_DIR / "test" / RUN_ID / f"test_ctc_loss_epoch_{best_epoch}.json"
test_ctc_loss = None

if test_loss_cache.is_file():
    test_ctc_loss = json.loads(test_loss_cache.read_text(encoding="utf-8")).get("test_ctc_loss")
elif (DRIVE_DATA_ROOT / "id_id" / "test.tsv").is_file():
    SAMPLE_RATE, N_FFT, HOP, N_MELS = 16000, 400, 160, 80

    def _norm(text):
        chars = unicodedata.normalize("NFC", text).casefold().replace("\u2019", "'")
        return " ".join("".join(c if c.isalnum() or c == "'" else " " for c in chars).split())

    def _mel_bank():
        def hz_to_mel(hz):
            return 2595 * math.log10(1 + hz / 700)
        mel_pts = torch.linspace(hz_to_mel(0), hz_to_mel(SAMPLE_RATE / 2), N_MELS + 2)
        left, center, right = (700 * (10 ** (mel_pts / 2595) - 1)).unfold(0, 3, 1).T
        fft_hz = torch.linspace(0, SAMPLE_RATE / 2, N_FFT // 2 + 1)
        rising = (fft_hz - left[:, None]) / (center - left)[:, None]
        falling = (right[:, None] - fft_hz) / (right - center)[:, None]
        return torch.minimum(rising, falling).clamp_min(0)

    _MEL = _mel_bank()
    _WIN = torch.hann_window(N_FFT)

    class _EvalModelA(nn.Module):
        def __init__(self, vocab_size, hidden, layers, channels):
            super().__init__()
            self.convs = nn.ModuleList([
                nn.Conv1d(N_MELS, channels[0], kernel_size=5, stride=2, padding=2),
                nn.Conv1d(channels[0], channels[1], kernel_size=3, stride=1, padding=1),
            ])
            self.gru = nn.GRU(channels[1], hidden, num_layers=layers,
                              batch_first=True, bidirectional=True,
                              dropout=0.1 if layers > 1 else 0)
            self.projection = nn.Linear(hidden * 2, vocab_size)

        def forward(self, features, lengths):
            features = features.transpose(1, 2)
            for conv in self.convs:
                features = torch.relu(conv(features))
                lengths = (lengths + 2 * conv.padding[0] - conv.kernel_size[0]) // conv.stride[0] + 1
                valid = torch.arange(features.size(-1), device=features.device) < lengths.to(features.device)[:, None]
                features = features.masked_fill(~valid[:, None, :], 0)
            packed = pack_padded_sequence(features.transpose(1, 2), lengths, batch_first=True, enforce_sorted=False)
            features, _ = pad_packed_sequence(self.gru(packed)[0], batch_first=True)
            return self.projection(features).log_softmax(-1), lengths

    vocab = state["settings"]["vocab"]
    char_to_id = {ch: idx for idx, ch in enumerate(vocab)}
    eval_model = _EvalModelA(len(vocab), **state["settings"]["model"]).to(DEVICE)
    eval_model.load_state_dict(state["model"])
    eval_model.eval()

    test_items = []
    for lang in ("id_id", "en_us"):
        tsv_path = DRIVE_DATA_ROOT / lang / "test.tsv"
        wav_dir = DRIVE_DATA_ROOT / lang / "audio" / "test"
        with tsv_path.open(encoding="utf-8-sig", newline="") as f:
            for fields in csv.reader(f, delimiter="\t", quoting=csv.QUOTE_NONE):
                if len(fields) == 7 and fields[0].isdigit():
                    wav_path = wav_dir / fields[1]
                    txt = _norm(fields[3])
                    if wav_path.is_file() and txt:
                        test_items.append((wav_path, txt))

    print(f"Menghitung Test CTC Loss pada {len(test_items)} sampel test (epoch {best_epoch})...", flush=True)
    losses = []
    with torch.inference_mode():
        for start in range(0, len(test_items), 32):
            chunk = test_items[start:start + 32]
            feats_list, targets_list = [], []
            for wav_path, txt in chunk:
                rate, pcm = wavfile.read(wav_path)
                samples = pcm.astype(np.float32)
                if np.issubdtype(pcm.dtype, np.integer):
                    info = np.iinfo(pcm.dtype)
                    samples = (samples - (info.max + 1) / 2) / ((info.max + 1) / 2) if info.min == 0 else samples / -info.min
                if samples.ndim == 2:
                    samples = samples.mean(axis=1)
                if rate != SAMPLE_RATE:
                    d = math.gcd(rate, SAMPLE_RATE)
                    samples = resample_poly(samples, SAMPLE_RATE // d, rate // d).astype(np.float32)
                power = torch.stft(torch.from_numpy(samples), N_FFT, HOP, window=_WIN, pad_mode="constant", return_complex=True).abs().square()
                mel = (_MEL @ power).clamp_min(1e-10).log().T.contiguous()
                target = [char_to_id[c] for c in txt if c in char_to_id]
                if target:
                    feats_list.append(mel)
                    targets_list.append(target)
            if not feats_list:
                continue
            lengths = torch.tensor([len(x) for x in feats_list], dtype=torch.long)
            padded = pad_sequence(feats_list, batch_first=True).to(DEVICE)
            preds, out_lens = eval_model(padded, lengths)
            joined = torch.tensor([t for seq in targets_list for t in seq], device=DEVICE)
            tgt_lens = torch.tensor([len(seq) for seq in targets_list])
            loss = nn.functional.ctc_loss(preds.transpose(0, 1), joined, out_lens, tgt_lens, zero_infinity=True)
            if torch.isfinite(loss):
                losses.append(loss.item())

    if losses:
        test_ctc_loss = sum(losses) / len(losses)
        test_loss_cache.parent.mkdir(parents=True, exist_ok=True)
        test_loss_cache.write_text(json.dumps({"epoch": best_epoch, "test_ctc_loss": round(test_ctc_loss, 4)}, indent=2), encoding="utf-8")

train_secs = [item["train_seconds"] for item in history if "train_seconds" in item]
elapsed_secs = [item["elapsed_seconds"] for item in history if "elapsed_seconds" in item]
vram_vals = [item["peak_vram_mib"] for item in history if item.get("peak_vram_mib") is not None]

def fmt_duration(seconds):
    return f"{seconds:,.1f} detik ({seconds / 60:.1f} menit)"

summary_lines = [
    "### Ringkasan Model A (From Scratch: CNN + BiGRU + CTC)",
    "",
    "| Aspek | Nilai |",
    "| --- | --- |",
    f"| Total parameters | {total_params:,} |",
    f"| Trainable parameters | {trainable_params:,} (100.0%) |",
    f"| Jumlah epoch tercatat | {len(history)} (checkpoint terbaik: epoch {best_epoch}) |",
]
if train_secs:
    summary_lines.append(f"| Rata-rata waktu training / epoch | {fmt_duration(sum(train_secs) / len(train_secs))} |")
    summary_lines.append(f"| Total waktu training ({len(train_secs)} epoch) | {fmt_duration(sum(train_secs))} |")
if elapsed_secs:
    summary_lines.append(f"| Rata-rata waktu total (train + dev) / epoch | {fmt_duration(sum(elapsed_secs) / len(elapsed_secs))} |")
    summary_lines.append(f"| Total waktu (train + dev, {len(elapsed_secs)} epoch) | {fmt_duration(sum(elapsed_secs))} |")
if vram_vals:
    summary_lines.append(f"| Puncak VRAM PyTorch (maks) | {max(vram_vals):,.1f} MiB |")
if test_ctc_loss is not None:
    summary_lines.append(f"| Test CTC Loss (Epoch {best_epoch}) | {test_ctc_loss:.3f} |")

display(Markdown("\n".join(summary_lines)))

lang_names = {"id_id": "Indonesia", "en_us": "Inggris"}
metric_lines = [
    f"### Metrik Checkpoint Terbaik (Epoch {best_epoch})",
    "",
    "| Split | Bahasa | WER | CER |",
    "| --- | --- | ---: | ---: |",
]
for lang in ("id_id", "en_us"):
    v = best_dev["per_language"][lang]
    metric_lines.append(f"| Dev | {lang_names[lang]} | {v['wer']:.1%} | {v['cer']:.1%} |")
metric_lines.append(f"| Dev (Rata-rata) | ID + EN | {best_dev['macro_wer']:.1%} | - |")

if test_saved is not None:
    for lang in ("id_id", "en_us"):
        v = test_saved["per_language"][lang]
        metric_lines.append(f"| Test eksploratif | {lang_names[lang]} | {v['wer']:.1%} | {v['cer']:.1%} |")
    metric_lines.append(f"| Test eksploratif (Rata-rata) | ID + EN | {test_saved['macro_wer']:.1%} | - |")

display(Markdown("\n".join(metric_lines)))

epochs = [item["epoch"] for item in history]
train_losses = [item.get("train_loss") for item in history]
val_losses = [item.get("val_loss") for item in history]
has_val_loss = any(v is not None for v in val_losses)

fig, (ax_loss, ax_err) = plt.subplots(1, 2, figsize=(15.5, 5.8), layout="constrained")

ax_loss.plot(epochs, train_losses, marker="o", markersize=4, color="#007C83", label="Training loss (CTC)")
if has_val_loss:
    ax_loss.plot(epochs, val_losses, marker="s", markersize=4, color="#D97706", label="Validation loss (CTC)")
if test_ctc_loss is not None:
    ax_loss.scatter(
        [best_epoch], [test_ctc_loss],
        marker="*", s=200, color="#DC2626", edgecolors="black", linewidths=0.6, zorder=6,
        label=f"Test CTC Loss (ep {best_epoch}: {test_ctc_loss:.2f})",
    )
ax_loss.axvline(best_epoch, color="#555555", linestyle=":", label=f"Terbaik: epoch {best_epoch}")
ax_loss.set_title("Training & Test CTC Loss", fontsize=12, fontweight="bold")
ax_loss.set_xlabel("Epoch", fontsize=10.5)
ax_loss.set_ylabel("CTC Loss", fontsize=10.5)
ax_loss.xaxis.set_major_locator(MaxNLocator(integer=True))
ax_loss.grid(alpha=0.25)
ax_loss.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=2, fontsize=9, frameon=True)

colors = {"id_id": "#007C83", "en_us": "#6D54AD"}
for lang in ("id_id", "en_us"):
    for metric, style in (("wer", "-"), ("cer", "--")):
        vals = [item["per_language"][lang][metric] * 100 for item in history]
        ax_err.plot(
            epochs, vals, linestyle=style, linewidth=1.8, color=colors[lang],
            label=f"Dev {lang_names[lang]} {metric.upper()}",
        )

if test_saved is not None:
    for lang, marker_wer, marker_cer in (("id_id", "*", "D"), ("en_us", "*", "D")):
        t_wer = test_saved["per_language"][lang]["wer"] * 100
        t_cer = test_saved["per_language"][lang]["cer"] * 100
        ax_err.scatter(
            [best_epoch], [t_wer], marker=marker_wer, s=180, color=colors[lang],
            edgecolors="black", linewidths=0.7, zorder=6,
            label=f"Test {lang_names[lang]} WER ({t_wer:.1f}%)",
        )
        ax_err.scatter(
            [best_epoch], [t_cer], marker=marker_cer, s=70, color=colors[lang],
            edgecolors="black", linewidths=0.7, zorder=6,
            label=f"Test {lang_names[lang]} CER ({t_cer:.1f}%)",
        )

ax_err.axvline(best_epoch, color="#555555", linestyle=":", label=f"Terbaik: epoch {best_epoch}")
ax_err.set_title("Validasi (Dev) & Test WER/CER per epoch", fontsize=12, fontweight="bold")
ax_err.set_xlabel("Epoch", fontsize=10.5)
ax_err.set_ylabel("Error (%)", fontsize=10.5)
ax_err.xaxis.set_major_locator(MaxNLocator(integer=True))
ax_err.grid(alpha=0.25)
ax_err.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3, fontsize=8.5, frameon=True)

plt.show()
if not has_val_loss:
    print("Catatan: validation loss per epoch belum dicatat; titik Test hanya dievaluasi pada checkpoint terbaik (epoch 25).")
